In [1]:
import pandas as pd
df = pd.read_csv("Telco_Customer_Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)
print(df.shape)

(7043, 20)


In [2]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["Churn"])
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("ti le churn - train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))

Train: (5634, 19) | Test: (1409, 19)
ti le churn - train: 0.2654 | test: 0.2654


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
binary_cols = ["SeniorCitizen"]
cat_cols = [c for c in X.columns if c not in num_cols + binary_cols]
print("So cot chu:", len(cat_cols))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ],
    remainder = "passthrough",
    verbose_feature_names_out = False,
)

So cot chu: 15


In [4]:
#xu ly tap train
preprocessor.fit(X_train)
feature_names = list(preprocessor.get_feature_names_out())
X_train_p = pd.DataFrame(preprocessor.transform(X_train), columns = feature_names, index=X_train.index)
X_test_p = pd.DataFrame(preprocessor.transform(X_test), columns = feature_names, index = X_test.index)
print("train:", X_train_p.shape, "|Test:", X_test_p.shape)
print("So gia tri thieu:", X_train_p.isna().sum(), X_test_p.isna().sum().sum())


train: (5634, 45) |Test: (1409, 45)
So gia tri thieu: tenure                                     0
MonthlyCharges                             0
TotalCharges                               0
gender_Female                              0
gender_Male                                0
Partner_No                                 0
Partner_Yes                                0
Dependents_No                              0
Dependents_Yes                             0
PhoneService_No                            0
PhoneService_Yes                           0
MultipleLines_No                           0
MultipleLines_No phone service             0
MultipleLines_Yes                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
InternetService_No                         0
OnlineSecurity_No                          0
OnlineSecurity_No internet service         0
OnlineSecurity_Yes                         0
OnlineBackup_No                            0
O

In [5]:
print("Train - trung binh:", X_train_p[num_cols].mean().round(3).to_dict())
print("Train - do lech chuan:", X_train_p[num_cols].std(ddof=0).round(3).to_dict())
print("Test -trung binh:", X_test_p[num_cols].mean().round(3).to_dict())

print(feature_names)

Train - trung binh: {'tenure': -0.0, 'MonthlyCharges': -0.0, 'TotalCharges': -0.0}
Train - do lech chuan: {'tenure': 1.0, 'MonthlyCharges': 1.0, 'TotalCharges': 1.0}
Test -trung binh: {'tenure': -0.023, 'MonthlyCharges': -0.028, 'TotalCharges': -0.043}
['tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'Stream

In [6]:
from sklearn.linear_model import LogisticRegression #mo dinh tuyen tinh don gian
from sklearn.ensemble import RandomForestClassifier # nhieu cay quyet dinh saong song
from xgboost import XGBClassifier # mo hinh nhieu cay quyet dinh noi tiep nhau

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000, #so vong lap toi da
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, #so cay trong rung
        max_depth=10, #do sau toi da cua moi cay
        min_samples_leaf=5, #moi la phai co it nhat 5 khach
        random_state=42,
        n_jobs=-1, #
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, # so cac cay them noi tiep nhau
        max_depth=4, 
        learning_rate=0.05, #moi cay chi sua mot phan nho cua cay trong giup hoc cham maf chawc
        subsample=0.8, #moi cay chi nhin 80% so dong
        colsample_bytree=0.8, #moi cay chi nhin 80% so cot
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    ),
}


In [7]:
for name, model in models.items():
    model.fit(X_train_p, y_train)
    print(name,"da train xong")

LogisticRegression da train xong
RandomForest da train xong
XGBoost da train xong


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
results = []
for name, model in models.items():
    proba = model.predict_proba(X_test_p)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })
results_df = pd.DataFrame(results).set_index("model").round(4)
results_df


,accuracy,precision,recall,f1,roc_auc
model,,,,,
LogisticRegression,0.8055,0.6572,0.5588,0.6040,0.8420
RandomForest,0.8062,0.6760,0.5187,0.5870,0.8420
XGBoost,0.7977,0.6498,0.5160,0.5753,0.8402


In [9]:
from sklearn.metrics import confusion_matrix

pred_lr = models["LogisticRegression"].predict(X_test_p)
print(confusion_matrix(y_test, pred_lr))
# ma tran nham lan
#dang [[doan dung "o lai", doan dung "roi di"]],
#       [bo sot khach roi di, doan dung "roi di"]
print("Accurcary neu luon doan 'o lai':", round(1-y_test.mean(), 4))

[[926 109]
 [165 209]]
Accurcary neu luon doan 'o lai': 0.7346


Chạy thử SHAP/LIME

In [12]:
import numpy as np
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer

model = models["XGBoost"]
i=1
x = X_test_p.iloc[[i]]
print(X_test.iloc[i][["tenure", "Contract", "InternetService", "MonthlyCharges"]].to_dict())
print("xac suat roi di:", round(model.predict_proba(x)[0, 1], 3), "| thuc te:", y_test.iloc[i])

{'tenure': 8, 'Contract': 'Month-to-month', 'InternetService': 'Fiber optic', 'MonthlyCharges': 100.15}
xac suat roi di: 0.913 | thuc te: 0


In [16]:
# TreeSHAP
# only for tree model
explainer_shap = shap.TreeExplainer(model)
shap_values = explainer_shap.shap_values(x)
# goi thanh series co ten feature, sap theo gia tri tuyetj doi giam dan
shap_series = pd.Series(shap_values[0], index=feature_names)
top8 = shap_series.reindex(shap_series.abs().sort_values(ascending=False).index).head(8)
print(top8.round(3))

#Check1: chay lai xem co ra y het khong/ true neu 2 mang bangnhau
print("chay lai giong het: ", np.allclose(shap_values, explainer_shap.shap_values(x)))

#Check2: gia tri nen + tong cac shap = dau ra cua model, tinhs theo log-odds
#expected_value la gia tri nen: dau ra trung binh cua model tren du lieu
base = float(np.ravel(explainer_shap.expected_value)[0])
logit = base + shap_values[0].sum()
print("nen:", round(base, 3), "| tong shap:", round(shap_values[0].sum(), 3), "| log-odds", round(logit, 3))

# ham sigmoid 1/ (1+e^-z) doi log-odds ve xac suat
print("xac suat: ", round(1/ (1+np.exp(-logit)), 3))

MonthlyCharges                 0.949
Contract_Month-to-month        0.591
tenure                         0.446
InternetService_Fiber optic    0.281
TotalCharges                   0.279
TechSupport_No                -0.196
MultipleLines_No               0.173
OnlineSecurity_No              0.161
dtype: float32
chay lai giong het:  True
nen: -1.039 | tong shap: 3.393 | log-odds 2.354
xac suat:  0.913


In [21]:
# tra ve trong so LIME cua khach so i, voi mot gia tri seed cho truoc
def lime_weights(i, seed):
    # tao bo giai thich vi LIME can du lieu train de biet moi feature thuong bien thien ra sao
    explainer = LimeTabularExplainer(
        training_data = X_train_p.values, #du lieu tham chieu dang mang so
        feature_names = feature_names,
        mode = "classification", #bai toan pha. loai
        discretize_continuous = False, #khong chi cot thanh khaongm giu nguyen gia tri
        random_state = seed, #seed cua phan ngau nhien
    )
    exp = explainer.explain_instance(
        X_test_p.values[i], #khach can giai thich
        #ham cho lime hoi model: nhan mang, tra ve xac suat 2 lop
        lambda a: model.predict_proba(pd.DataFrame(a, columns=feature_names)),
        num_features=len(feature_names), #lay trong so cua 45 feature
        num_samples=1000, #tao 1000 khach gia xung quanh khach that
    )
    weights = dict(exp.as_map()[1])
    # doi thanh series theo dung thu tu 45 cot. cot khong co trong weightsthi la 0
    return pd.Series([weights.get(k, 0.0) for k in range(len(feature_names))], index=feature_names)
# chay voi 3 seed va in 5 feature quan trong nhat cua moi lan
for seed in [0, 1, 2]:
    w = lime_weights(i, seed)
    print("seed ", seed)
    print(w.reindex(w.abs().sort_values(ascending=False).index).head(5).round(4))

seed  0
Contract_Month-to-month        0.0327
tenure                        -0.0302
MonthlyCharges                 0.0290
InternetService_Fiber optic    0.0134
TotalCharges                  -0.0122
dtype: float64
seed  1
Contract_Month-to-month        0.0343
tenure                        -0.0342
MonthlyCharges                 0.0310
TotalCharges                  -0.0204
InternetService_Fiber optic    0.0142
dtype: float64
seed  2
Contract_Month-to-month    0.0332
MonthlyCharges             0.0299
tenure                    -0.0287
TotalCharges              -0.0143
OnlineSecurity_No          0.0111
dtype: float64


cungf một khách cùng một model cùng phương pháp LIME, chỉ dổi seed
phần ổn định 3 feature đầu Contract_Month-to-month (có măt ở 3 lần chạy luôn đứng hạng 1), tenure, MonthlyCharges          

In [23]:
# thử LIME trên 10 khách, đếm số feature trùng nhau giữa các seed
rows = []
for j in range(10):
    tops = []
    for seed in [0, 1, 2]:
        w = lime_weights(j, seed)
        tops.append(set(w.abs().sort_values(ascending=False).head(5).index))
    rows.append({"khach":j,
                 "seed 0-1": len(tops[0] & tops[1]),
                 "seed 0-2": len(tops[0] & tops[2]),
                 "seed 1-2": len(tops[1] & tops[2])})
pd.DataFrame(rows).set_index("khach")

,seed 0-1,seed 0-2,seed 1-2
khach,,,
0,5,4,4
1,5,4,4
2,5,4,4
3,4,4,5
4,4,4,4
5,5,4,4
6,5,4,4
7,5,4,4
8,4,4,5


data can perfect or kem. tuy thuoc vao cach do
+ mot giai thich tot phai vung (đầu vào giống nhau thì giải thích cũng phải giống nhau)
+data có nhiều biến phân loại -> thêm nhiễu liên tục không còn nhiều ý nghĩa -> nên dùng bản rời rạc
+ khong có gia trị lý tưởng tuyệt đối mà nên so tương đối giữa các phương pháp

In [30]:
import numpy as np

# Hai dòng này giúp notebook tự nạp lại file .py khi bạn sửa nó, khỏi phải Restart.
%load_ext autoreload
%autoreload 2

from stability_metrics import jaccard_topk, spearman_rank, cosine_sim, explanation_lipschits

# Một vector attribution nhỏ (6 feature) làm mốc để so sánh.
a = np.array([0.5, -0.4, 0.3, 0.1, 0.0, 0.05])

# Bốn trường hợp, mỗi trường hợp là một "lần chạy thứ hai" khác nhau của cùng lời giải thích.
cases = {
    "giống hệt": a,
    "đổi dấu toàn bộ": -a,
    "nhân 10 lần": 10 * a,
    "đổi chỗ 2 giá trị": np.array([0.5, -0.4, 0.0, 0.1, 0.3, 0.05]),
}
for name, b in cases.items():
    # :20s căn tên rộng 20 ký tự, :.2f làm tròn 2 chữ số thập phân cho dễ đọc.
    print(f"{name:20s} jaccard_top3={jaccard_topk(a, b, 3):.2f}  "
          f"spearman={spearman_rank(a, b):.2f}  cosine={cosine_sim(a, b):.2f}")

# Lipschitz-like: hai giải thích dài 1 nhưng khác nhau, đầu vào dịch chuyển 0,1.
xi, xj = np.array([0., 0., 0.]), np.array([0.1, 0., 0.])
print(explanation_lipschits(np.array([1., 0.]), np.array([0.8, 0.6]), xi, xj))   # khác nhau
print(explanation_lipschits(np.array([1., 0.]), np.array([1., 0.]), xi, xj))     # giống nhau

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
giống hệt            jaccard_top3=1.00  spearman=1.00  cosine=1.00
đổi dấu toàn bộ      jaccard_top3=1.00  spearman=1.00  cosine=-1.00
nhân 10 lần          jaccard_top3=1.00  spearman=1.00  cosine=1.00
đổi chỗ 2 giá trị    jaccard_top3=0.50  spearman=0.49  cosine=0.82
6.324555320336758
0.0


In [33]:
import pandas as pd
from stability_metrics import stability_report

reports = []
for j in range(10):                                    # 10 khách đầu của tập test
    # Mỗi hàng là vector LIME (45 số) của một seed. np.stack ghép 3 hàng thành mảng 3 x 45.
    runs = np.stack([lime_weights(j, s).values for s in [0, 1, 2]])
    # Tính điểm ổn định của khách j, trung bình trên 3 cặp seed.
    reports.append(stability_report(runs, ks=(3, 5, 10)))

report_df = pd.DataFrame(reports)     # bảng 10 khách x 5 thước đo
print(report_df.round(3))             # điểm từng khách
print(report_df.mean().round(3))      # điểm trung bình trên 10 khách

   jaccard_top3  jaccard_top5  jaccard_top10  spearman  cosine
0           1.0         0.778          0.624     0.473   0.938
1           1.0         0.778          0.632     0.541   0.950
2           1.0         0.778          0.624     0.440   0.932
3           1.0         0.778          0.558     0.408   0.933
4           1.0         0.667          0.624     0.455   0.938
5           1.0         0.778          0.632     0.491   0.944
6           1.0         0.778          0.674     0.451   0.941
7           1.0         0.778          0.632     0.386   0.931
8           1.0         0.778          0.581     0.387   0.931
9           1.0         1.000          0.667     0.417   0.939
jaccard_top3     1.000
jaccard_top5     0.789
jaccard_top10    0.625
spearman         0.445
cosine           0.938
dtype: float64


In [40]:
# kiem tra time cho mot lan chay thu
#seed co dieu khien duoc phan ngau nhien khong
import time 
from explainers import make_background, lime_attribution, kernelshap_attribution, exact_shap_attribution

background = make_background(X_train_p, n=30)
columns = list(X_train_p.columns)
x = X_test_p.values[1]

#kiem tra seed-> cung phai giong, khac phai khac
for name, m in models.items():
    k1 = kernelshap_attribution(m, background, columns, x, seed=0)
    k2 = kernelshap_attribution(m, background, columns, x, seed=0)
    k3 = kernelshap_attribution(m, background, columns, x, seed=1)
    l1 = lime_attribution(m, X_train_p, x, seed=0)
    l2 = lime_attribution(m, X_train_p, x, seed=0)
    l3 = lime_attribution(m, X_train_p, x, seed=1)
    print(name, "kernelSHAP cung seed giong: ", np.allclose(k1, k2), "khac seed khac: ", not np.allclose(k1,k3),
          "LIME cung seed giong: ", np.allclose(l1, l2), "khac LIME khac: ", not np.allclose(l1, l3))

#do avegare time chua 1 lan giai thich

rows = []        # (1) BẮT BUỘC: tạo danh sách rỗng mới mỗi lần chạy, để không dính dòng cũ
for name, m in models.items():
    start = time.time()                    # (2) ghi lại thời điểm bắt đầu, ngay trước khối cần đo
    for s in range(3):                     # chạy LIME 3 lần với seed 0, 1, 2
        lime_attribution(m, X_train_p, x, seed=s)
    t_lime = (time.time() - start) / 3     # thời gian đã trôi qua, chia 3 để ra trung bình một lần

    start = time.time()                    # đặt lại mốc bắt đầu cho phép đo tiếp theo
    for s in range(3):
        kernelshap_attribution(m, background, columns, x, seed=s)
    t_ks = (time.time() - start) / 3

    start = time.time()
    exact_shap_attribution(m, X_train_p, x)
    t_exact = time.time() - start

    # (3) Tên cột viết đúng: "KernelSHAP" (chữ K hoa), khớp với các ô ước tính ở sau.
    rows.append({"model": name, "LIME": t_lime, "KernelSHAP": t_ks, "SHAP chính xác": t_exact})

timing = pd.DataFrame(rows).set_index("model")
print(timing.shape)             # kỳ vọng: (3, 3)  (3 model, 3 cột)
print(timing.round(3))

# Ước tính tổng thời gian: cộng LIME và KernelSHAP của từng model, .sum() cộng 3 model lại,
# rồi nhân với số khách và số seed.
for n_cust, n_seeds in [(100, 10), (200, 20)]:
    total = (timing["LIME"] + timing["KernelSHAP"]).sum() * n_cust * n_seeds
    print(f"{n_cust} khách x {n_seeds} seed: khoảng {total / 60:.1f} phút")

LogisticRegression kernelSHAP cung seed giong:  True khac seed khac:  True LIME cung seed giong:  True khac LIME khac:  True
RandomForest kernelSHAP cung seed giong:  True khac seed khac:  True LIME cung seed giong:  True khac LIME khac:  True
XGBoost kernelSHAP cung seed giong:  True khac seed khac:  True LIME cung seed giong:  True khac LIME khac:  True
(3, 3)
                     LIME  KernelSHAP  SHAP chính xác
model                                                
LogisticRegression  0.005       0.006           0.000
RandomForest        0.044       0.099           0.023
XGBoost             0.009       0.021           0.022
100 khách x 10 seed: khoảng 3.1 phút
200 khách x 20 seed: khoảng 12.3 phút


In [ ]:
from sklearn.neighbors import NearestNeighbors

#kho tim hang xom: toan bo khach (train+test), da tien xu ly
pool = pd.concat([X_train_p, X_test_p])

model = models["XGBoost"]
p_pool = model.predict_proba(pool)[:, 1] # xac xua roi di cua moi khach trong kho
p_test = model.predict_proba(X_test_p)[:, 1] # xac xua roi di cua cac khach can giai thich

#nearesneighnors : voi moi khach test tim 20 khach hang gan nhat trong kho theo khoang chach euclid
K = 20
nn = NearestNeighbors(n_neighbors = K+1).fit(pool.values)
dist, idx = nn.kneighbors(X_test_p.values) 

DP_MAX = 0.05
rows = []
for eps in [0.5, 1.0, 1.4]:
    counts = [] # so hang xom hop le cua tung khach
    for r in range(len(X_test_p)):
        ok = ((dist[r] > 1e-9)
              & (dist[r] <= eps)
              & (np.abs(p_pool[idx[r]] - p_test[r]) <= DP_MAX)) # du doan gan nhu khong doi
        counts.append(ok.sum())
    counts = np.array(counts)
    rows.append({"eps": eps,
                 "co >=1 hang xom": (counts >= 1).mean(),
                 "co >=3 hang xom": (counts >=3).mean(),
                 "trung binh": counts.mean()})

print(pd.DataFrame(rows).round(3))
print("khoang cach toi hang xom gan nhat: ", np.percentile(dist[:, 1], [10, 50, 90]).round(2))

   eps  co >=1 hang xom  co >=3 hang xom  trung binh
0  0.5            0.352            0.150       1.331
1  1.0            0.380            0.180       1.781
2  1.4            0.442            0.224       2.159
khang cach toi hang xom gan nhat:  [0.04 1.15 2.  ]
